# Model Reduction

*Model reduction* is the process of deriving a simplified model of a system that can be solved faster, known as a reduced order model (ROM). There are many ways of doing this, some of which are "intrusive" (manipulating the governing equations of the system) and some are "non-intrusive" (requiring only data from - i.e. solutions of - the system).

Here we will focus on one particular technique, *projection reduced order models* (PROMs), that start from a complicated set of equations from a *full order model* (FOM), and project them onto a lower-dimensional subspace of solutions to derive a smaller ROM. Here we will consider a PROM known as POD-Galerkin projection with DEIM hyperreduction (POD-Galerkin-DEIM).

## Full order model (FOM)

Let us build a few FOMs so we can derive ROMs from them. We will work with partial differential equations (PDEs), although PROMs in general can be constructed from any ODE (and a spatially-discretized PDE is just a system of ODEs, as we will see).

### Diffusion equation (1D): A linear PDE

The diffusion or heat equation is
$$\frac{\partial u}{\partial t} = \kappa \nabla^2 u \equiv \Delta u$$
where the $\kappa$ parameter is the diffusion rate coefficient. It describes the diffusion of heat or a substance like a chemical throughout a spatial domain over time; $u(x,t)$ &mdash; which I will often write as $u(t)$ where we are thinking of $u$ as a vector-valued function of time, but sometimes will write as $u(x)$ when I am thinking of the variable at a given location and at a fixed time &mdash; could be temperature, chemical concentration, etc.

The Laplacian operator is
$$\Delta = \nabla^2 = \nabla\cdot\nabla$$
which in 1D reduces to the spatial second derivative
$$\nabla^2 = \frac{\partial}{\partial x}\left(\frac{\partial}{\partial x}\right) = \frac{\partial^2}{\partial x^2}$$

To compute $\nabla^2 u$ numerically on a computer, we will spatially discretize the inner first-derivative operator $\partial/\partial x$ by centered finite differences (here $u\equiv u(t)$ is the state at a given time):
$$\frac{\partial u}{\partial x} \approx \frac{u(x+\delta)-u(x-\delta)}{2\delta}$$
Then discretizing the outer first-derivative and applying it to the first derivative, $\partial(\partial/\partial x)/\partial x$,
$$\frac{\partial}{\partial x}\left(\frac{\partial u}{\partial x}\right) \approx \frac{\left(\frac{u(x+2\delta)-u(x)}{2\delta}\right) - \left(\frac{u(x)-u(x-2\delta)}{2\delta}\right)}{2\delta}$$
or
$$\nabla^2 u \approx \frac{u(x+2\delta) - 2u(x) + u(x-2\delta)}{4\delta^2}$$
which under a redefinition $\delta\rightarrow\delta/2$ is
$$\nabla^2 u \approx \frac{u(x+\delta) - 2u(x) + u(x-\delta)}{\delta^2}$$
We think of the Laplacian operator as computing something proportional to the difference between a point and the average of its neighbors
$$\nabla^2 u \approx -\frac{2}{\delta^2}\left(u(x) - [u(x+\delta)+u(x-\delta)]/2\right) = -\frac{2}{\delta^2}\left(u - \bar{u}\right)$$
which generalizes to $\nabla^2 u \approx -\frac{2d}{\delta^2}\left(u - \bar{u}\right)$ in $d$ dimensions, and is what gives it an interpretation as a diffusive process, because the heat (or other variable) spreads out in a way that "relaxes" the solution in each neighborhood towards its mean.

To complete our discretization, we also discretize in time using a forward finite-difference Euler scheme:
$$\frac{\partial u}{\partial t} \approx \frac{u(t+\Delta t)-u(t)}{\Delta t}$$
leading to the Euler timestepper
$$u(t+\Delta t) \approx u(t) + \frac{\partial u}{\partial t} \Delta t$$
which, for the 1D diffusion equation, becomes
$$u(x,t+\Delta t) \approx u(x,t) + \left[\frac{\kappa}{\delta^2} (u(x+\delta,t) - 2u(x,t) + u(x-\delta,t))\right] \Delta t$$

We can implement this in code, with "absorbing" Dirichlet boundary conditions fixing the endpoints to 0. In modular coding fashion, we separate out a function to compute the right hand side (RHS), $\kappa \nabla^2 u$, of the differential equation and call it from an Euler timestepper function.

In [1]:
using LinearAlgebra
using SparseArrays
using Statistics
using Plots

In [2]:
function fom_diffusion!(du, u, t, (κ,δ)=p)
    for i = 2:length(u)-1
        du[i] = κ * (u[i+1] - 2*u[i] + u[i-1]) / δ^2
    end
end

fom_diffusion! (generic function with 2 methods)

In [3]:
function solve_euler(rhs!, u₀, ts, p)
    N = length(u₀)
    Nsteps = length(ts)
    Δt = ts[2]-ts[1]
    u = zeros(N, Nsteps)
    u[:,1] = u₀
    duₜdt = zeros(N)
    
    for t = 1:Nsteps-1
        uₜ = @view u[:,t]
        rhs!(duₜdt, uₜ, ts[t], p) # update duₜ (= ∂u/∂t) array in-place by overwriting (mutating)
        u[:,t+1] = u[:,t] + duₜdt * Δt
    end

    u
end

solve_euler (generic function with 1 method)

In [ ]:
κ = 0.1

T = 1.0
Nsteps = 65536
ts = LinRange(0, T, Nsteps)
Δt = ts[2]-ts[1]

L = 1.
Nx = 512
Δx = L/Nx

x = LinRange(0, L, Nx)
u₀ = sin.(π*x);

In [5]:
p = (κ, Δx)
@time u_heat = solve_euler(fom_diffusion!, u₀, ts, p);
@time u_heat = solve_euler(fom_diffusion!, u₀, ts, p);

  0.462482 seconds (683.82 k allocations: 1.016 GiB, 66.92% gc time, 9.74% compilation time)
  0.282052 seconds (589.83 k allocations: 1.012 GiB, 57.50% gc time)


In [ ]:
@time @gif for i = 1:length(ts)
    plot(x, u_heat[:,i], title="Heat equation", ylim=(0.0, maximum(u₀)+1e-3), lw=3, label="")
end every 1024

### Matrix form of 1D diffusion equation

For model reduction it may be helpful to rewrite in terms of matrices. The spatially discretized PDE
$$\frac{\partial u_i}{\partial t} = \kappa (u_{i+1} - 2u_i + u_{i-1})/\delta^2$$
is actually just a sparse coupled system of ODEs for the variables $u=\{u_1,\ldots,u_N\}$. Since it is linear, we can write this in matrix form as
$$\frac{\partial u}{\partial t} = Au$$
where
$$A = \frac{\kappa}{\delta^2}
\begin{bmatrix}
-2 & 1 & 0 & \cdots & 0 \\
1 & -2 & 1 & \ddots & \vdots \\
0 & 1 & -2 & \ddots & 0 \\
\vdots & \ddots & \ddots & \ddots & 1 \\
0 & \cdots & 0 & 1 & -2
\end{bmatrix}$$

We can write the new RHS and build the $A$ matrix by specifying the three nonzero diagonals using sparse array syntax.

In [ ]:
function fom_diffusion_matrix!(du, u, t, A=p)
    du .= A * u
end

In [ ]:
A = (κ / Δx^2) * spdiagm(-1 => fill(1.0,Nx-1), 0 => fill(-2.0,Nx), 1 => fill(1.0,Nx-1))

In [ ]:
@time u_heat = solve_euler(fom_diffusion_matrix!, u₀, ts, A);
@time u_heat = solve_euler(fom_diffusion_matrix!, u₀, ts, A);

In [ ]:
@time @gif for i = 1:length(ts)
    plot(x, u_heat[:,i], title="Heat equation", ylim=(0.0, maximum(u₀)+1e-3), lw=3, label="")
end every 1024

### Allen-Cahn equation (1D): A nonlinear PDE

The diffusion/heat equation is linear. We would also like to consider an exmaple of a nonlinear FOM. The Allen-Cahn equation is a simple extension, which is the diffusion equation plus a nonlinear term:
$$\frac{\partial u}{\partial t} = \nabla^2 u + f(u)$$
It is common to take as default a cubic nonlinearity:
$$f(u) = u^3 - u$$
This equation describes a reaction-diffusion-like equation, but rather than describing the spatial dynamics of chemically reacting species, it is more commonly used to model phenomena like grain boundary dynamics in polycrystalline metals, where $u$ represents how close the system is to one phase or another, and $f(u)$ is the gradient of a free energy functional (such as a quartic double-well for a two-phase system).

Again, for ROMs it will be convenient to treat the linear term in matrix form; the linear part is the same as the diffusion equation (the Laplacian), so we can reuse the matrix.

In [ ]:
function fom_allen_cahn_matrix!(du, u, t, (A,f)=p)
    du .= A * u + f.(u)
end

In [ ]:
f(u) = u^3 - u

In [ ]:
@time u_ac = solve_euler(fom_allen_cahn_matrix!, u₀, ts, (A,f));
@time u_ac = solve_euler(fom_allen_cahn_matrix!, u₀, ts, (A,f));

In [ ]:
@time @gif for i = 1:length(ts)
    plot(title="Allen-Cahn equation vs. Heat equation", ylim=(0.0, maximum(u₀)+1e-3))
    plot!(x, u_heat[:,i], label="Heat", lw=3)
    plot!(x, u_ac[:,i], ylim=(0.0, maximum(u₀)+1e-3), lw=3, label="Allen-Cahn")
end every 1024

## Reduced order model (ROM)

With some FOMs in hand to reduce, we now consider projection-based reduced order models (PROMs).

Consider again the matrix form of the spatially-discretized PDE (or equivalently a linear ODE system),
$$\frac{\partial u}{\partial t} = Au$$

For a high-resolution grid, the dimension $N$ of the state vector $u$ can be large, and solving this equation is expensive (still of $O(N)$ even for a sparse matrix).

Instead, consider representing the state vector at any time $u\equiv u(t)$ as a linear combination of $r$ basis vectors,
$$u = \sum_{i=1}^r \tilde{u}_i U_i = U\tilde{u}$$
where $U = [U_1, \ldots, U_r]$ is a $N\times r$ matrix whose columns are the basis vectors $\{U_i\}$ and $\tilde{u} = [\tilde{u}_1,\ldots,\tilde{u}_r]$ is the $r$-dimensional vector of basis coefficients $\{\tilde{u}_i\}$.

If the solution of the FOM can be expanded in a relatively low-dimensional basis ($r \ll N$), then we can hope to solve an ODE system of $r$ variables giving the dynamics of the *coefficients* $\tilde u$, instead of the much larger system of the $N$ state variables $u$.

### Galerkin projection

We can think of $U:\mathbb{R}^r\rightarrow\mathbb{R}^N$ as a "lifting" matrix that maps or reconstructs a reduced state vector to the corresponding full-order state vector, $u = U\tilde{u}$. Left-multiplying both sides by $U'$, we get $U'u = U'U\tilde{u}$. If the basis is *orthonormal*, $U'U = I$. Therefore $U'u = (U'U)\tilde{u} = \tilde{u}$. Thus, $U'$ is a "projection" matrix that maps a full-order state vector to its reduced vector form, $\tilde{u} = U'u$.

Remember these central relationships &mdash; we will use them everywhere:
1. *Reduction / Projection* (full &rarr; reduced): $\tilde{u} = U'u$
2. *Reconstruction / Lifting* (reduced &rarr; full): $u = U\tilde{u}$

Consider again a linear matrix ODE (which could be a discretized linear PDE):
$$\frac{\partial u}{\partial t} = Au$$

If we replace $u$ with its basis expansion $U\tilde{u}$, we can rewrite the FOM as:
$$\frac{\partial(U\tilde{u})}{\partial t} = A(U\tilde{u})$$

Now, project onto the reduced basis by multiplying both sides by $U'$:
$$U'\frac{\partial(U\tilde{u})}{\partial t} = U'AU\tilde{u}$$

Since $U$ is time-independent, we can bring it outside the derivative:
$$U'U\frac{\partial\tilde u}{\partial t} = U'AU\tilde{u}$$

Since $U$ is an orthonormal basis, $U'U = I$, so
$$\frac{\partial\tilde u}{\partial t} = (U'AU)\tilde{u}$$

We now have a *reduced ODE* written purely in terms of the reduced state variables,
$$\frac{\partial\tilde u}{\partial t} = \tilde{A}\tilde{u}$$
where $\tilde{A} = U'AU$ is a $r\times r$ matrix acting on an $r$-dimensional reduced state vector $\tilde{u}$, which can be *precomputed* as a matrix multiplication before solving the ODE.

We have therefore reduced the large $N$-dimensional dynamical system of state variables to a much smaller $r$-dimensional dynamical system of reduced basis coefficients. If we want to reconstruct the solution $u\equiv u(t)$ at any time, we just left-multiply the state vector by $U$, $u = U\tilde{u}$.

This procedure of reducing a set of equations is called *Galerkin projection*. Here, we have used a *global* basis (expanding the entire state vector in a basis), which is commonly used in PROMs. If, instead, we used a *local* basis &mdash; where every *grid cell* can be thought of as containing a spatially varying solution expanded in basis functions inside the grid cell, so we have a different vector of coefficients for each grid cell &mdash; we are on the road to the *finite element method* (FEM) for solving PDEs.

## Dimension reduction: Proper orthogonal decomposition (POD)

How do we find a good basis $U$ for the solution of our FOM? We could try some typical basis for expanding functions like the Fourier basis, or spherical harmonics, etc. However, many FOMs do not have a convenient low-dimensional expansion in these terms.

Instead, it is common to use a *data-driven* basis. Suppose we have some training data consisting of a $N\times M$ matrix $X$ of $M$ "snapshots" ($N$-dimensional state vectors) at different times from one or more FOM solutions, $X = [X_1,\ldots,X_M]$.

From snapshot data $X$, a procedure known as *proper orthogonal decomposition (POD)* extracts a linear basis $U$ that is *optimal* in two senses:

1. *Minimum reconstruction error*: If we project onto a low-dimensional basis and then lift or reconstruct back to the full space, we will not recover the original vector: projection into a lower-dimensional subspace is a lossy compression or approximation. The $L^2$ error in approximating a vector $x$ by a basis $U$ is $\Vert UU'x - x\Vert_2^2$. The Eckart-Young-Mirsky theorem states that proper orthogonal decomposition of X gives the $U$ that minimizes sum of such reconstruction errors over all the snapshots $x\in \{X_i\}$.

2. *Maximum variance explained*: We can construct the (centered or mean-subtracted) covariance matrix of the data, $C = \frac{1}{M}XX'$. The POD basis $U$ corresponds to the eigenvectors of $C$; if we sort them in terms of decreasing eigenvalue, then the eigenvalue corresponding to each $U_i$ is the variance of each principal eigenvector; this is the same as *principal component analysis (PCA)*. The "variance explained" by $U$ (the first $r$ eigenvectors of $C$) is the sum of the variances of the $\{U_i\}$, and the POD basis $U$ maximizes the variance of X explained by $U$, out of any $r$ basis vectors you could choose.

How do you find this optimal POD basis $U$? The second point indicates that you can find it by an eigendecomposition of the snapshot data matrix $X$. However, in practice, we use a *singular value decomposition (SVD)*,
$$X = U\Sigma V'$$
where $U$ and $V$ are orthogonal and $\Sigma$ is diagonal; the diagonal elements, or *singular values*, of $\Sigma$ are the standard deviations of the eigenvectors of the covariance matrix (divided by a normalization $\sqrt{M}$).

In practice we want to *truncate* the SVD basis: the $U$ factor in SVD gives $\mathrm{rank}\,X \leq \min(N,M)$ basis vectors, and for practicality we may chose the first $r$ of them where $r$ is chosen to be computationally tractable.

Let us construct a snapshot matrix from the solution of a FOM and study its singular values $S$ (diagonal of $\Sigma$) and left and right singular vectors $U$ and $V$ and the approximation quality of this basis.

In [ ]:
X = u_heat
@time U, S, V = svd(X);

The singular values are sorted in decreasing order; plot the first 10.

In [ ]:
plot(S, xlim=(1,10), xticks=1:10, lw=3, xlabel="POD basis vector", ylabel="Singular value", title="Singular values of heat equation snapshots", label="")

We see that the first basis vector is by far the most important. We can also see this in "fraction of cumulative variance explained", $\eta_i = \left(\sum_{j=1}^i \sigma_i^2\right) / \left(\sum_{j=1}^r \sigma_i^2\right)$ where $\sigma_i = S_i/\sqrt{M}$.

In [ ]:
M = size(X,2) # here # snapshots = # timesteps (Nsteps)
σ = S/√M
η = cumsum(σ.^2)/sum(σ.^2)
plot(η, xlim=(1,10), xticks=1:10, lw=3, xlabel="POD basis vector", ylabel="Fraction cumulative variance explained", title="Fractional cumulative variance explained", label="")

We see that the first POD basis vector explains 99.9999% of the variance in the solution to the heat equation (across snapshots, i.e. across time)! This is special to heat-like equations, whose eigenspectrum is known to decay exponentially quickly. (This is, in fact, why we chose to start with the heat equation to study PROMs - it is the best-case example.)

This is related to the concept of the Kolmogorov $n$-width, which is the smallest possible worst-case approximation error you can get from a $n$-dimensional subspace, which equals the singular value $\sigma_{n+1}$. We say that for the heat equation, the Kolmogorov $n$-width decays exponentially.

In short, it's better to view the singular values of solutions to the heat equation on a log scale.

In [ ]:
plot(S, yscale=:log10, xlim=(1,10), ylim=(1e-3,1e4), xticks=1:10, lw=3, xlabel="POD basis vector", ylabel="Singular value", title="Singular values of heat equation snapshots", label="")

What does the POD basis look like? We'll plot the first few orthonormal modes (left singular vectors) of $U$.

Reminder: POD modes are data-dependent and depend on the particular solution(s) we're using. If we used different initial conditions, or solved for different times, we'd get different modes. They're not solution-independent eigenfunctions of the heat equation's Laplace operator itself, though the POD modes can of course be expanded in the Laplacian eigemodes (and will also depend on the initial condition of the solution that generated the snapshot data).

In [ ]:
plt = plot(legend=:bottom, title="POD modes of heat equation snapshots")
for i = 1:4
    plot!(plt, U[:,i], lw=3, label="Mode $i")
end
plt

How well does the first basis vector reconstruct the whole solution? Our $U$ basis will be the first $r$ columns (here $r=1$) of the $U$ matrix from SVD, so we will compare the true $u$ to the projected-and-reconstructed $U_{\cdot,1\ldots r} U'_{\cdot,1\ldots r} u$. 

In [ ]:
r = 1

@time @gif for i = 1:length(ts)
    plot(x, u_heat[:,i], legend=:bottom, label="FOM snapshots", title="Heat equation", ylim=(0.0, maximum(u₀)+1e-3), lw=3)
    plot!(x, U[:,1:r]*U[:,1:r]'*u_heat[:,i], ls=:dash, label="POD reconstruction (1 mode)", lw=3)
end every 1024

What about for the Allen-Cahn equation?

In [ ]:
X = u_ac
@time U, S, V = svd(X)

plot(S, yscale=:log10, xlim=(1,10), ylim=(1e-3,1e4), xticks=1:10, lw=3, xlabel="POD basis vector", ylabel="Singular value", title="(log scale) Singular values of Allen-Cahn snapshots", label="")

In [ ]:
plt = plot(legend=:bottom, title="POD modes of Allen-Cahn snapshots")
for i = 1:4
    plot!(plt, U[:,i], lw=3, label="Mode $i")
end
plt

In [ ]:
r = 1

@time @gif for i = 1:length(ts)
    plot(x, u_ac[:,i], legend=:bottom, label="FOM snapshots", title="Allen-Cahn equation", ylim=(0.0, maximum(u₀)+1e-3), lw=3)
    plot!(x, U[:,1:r]*U[:,1:r]'*u_ac[:,i], ls=:dash, label="POD reconstruction (1 mode)", lw=3)
end every 1024

The situation is very similar.

## POD-Galerkin ROM of the diffusion equation

Now that we have a good reduced basis, we are ready to construct a ROM through POD-Galerkin projection.

Recall that the linear diffusion or heat equation FOM can be expressed in matrix form as:
$$\frac{\partial u}{\partial t} = Au$$
for a particular $N\times N$ matrix $A$ arising from the finite-difference discretized Laplacian operator, where $N$ is the dimension of the state vector (number of grid cells).

If we have a $N\times r$ reduced basis matrix $U$ represnting $r$ basis vectors, it can be used to reconstruct an $r$-dimensional reduced state vector $\tilde u$ to an $N$-dimensional full state vector $u$. We can then project the FOM onto the $U$ basis to produce a ROM which be expressed as a similar, but lower-dimensional matrix equation
$$\frac{\partial \tilde u}{\partial t} = \tilde{A}\tilde{u}$$
where $\tilde{A} = U'AU$ is $r\times r$.

We will implement this for the heat equation, beginning with $r=1$ POD mode. We will change notation so $U_{SVD}$ is the left singular vectors from the singular value decomposition, and $U$ is the first $r$ columns (leading singular vectors), where we choose $r$ to fix the size of the ROM.

In [ ]:
U_SVD_heat,_,_ = svd(u_heat)

r = 1
U = U_SVD_heat[:,1:r]

Ã = U'*(A*U) # [r x r]

In [ ]:
function rom_diffusion_matrix!(dũ, ũ, t, Ã=p)
    dũ .= Ã * ũ
end

In [ ]:
@time ũ_heat = solve_euler(rom_diffusion_matrix!, U'u₀, ts, Ã);
@time ũ_heat = solve_euler(rom_diffusion_matrix!, U'u₀, ts, Ã);

In [ ]:
@time @gif for i = 1:length(ts)
    plot(x, u_heat[:,i], title="Heat equation", ylim=(0.0, maximum(u₀)+1e-3), lw=3, label="FOM")
    plot!(x, U*ũ_heat[:,i], ls=:dash, lw=3, label="ROM", )    
end every 1024

The solution is identical (in the eyeball norm). On my computer, the ROM solves 40$\times$ faster than the sparse matrix FOM, and 20$\times$ faster than the direct for-loop stencil FOM, for $N$=512. This is because we went from solving an ODE system of 512 variables to an ODE of 1 variable (for $r=1$ POD modes).

Let us plot the residual errors to see how good they are.

In [ ]:
@time @gif for i = 1:length(ts)
    plot(x, U*ũ_heat[:,i] - u_heat[:,i], title="Heat equation residuals", ylim=(-1e-2, 1e-2), lw=3, label="ROM-FOM")
end every 1024

Now let's try adding a second POD mode, even though it matters much much less than the first.

In [ ]:
r = 2
U2 = U_SVD_heat[:,1:r]
Ã2 = U2'*(A*U2)
@time ũ_heat2 = solve_euler(rom_diffusion_matrix!, U2'u₀, ts, Ã2)
@time ũ_heat2 = solve_euler(rom_diffusion_matrix!, U2'u₀, ts, Ã2)

In [ ]:
@time @gif for i = 1:length(ts)
    plot(x, U*ũ_heat[:,i] - u_heat[:,i], title="Heat equation residuals", ylim=(-1e-2, 1e-2), lw=3, label="1 mode")
    plot!(x, U2*ũ_heat2[:,i] - u_heat[:,i], title="Heat equation residuals", ylim=(-1e-2, 1e-2), lw=3, label="2 modes")
end every 1024

In [ ]:
rmse(x,y) = √mean(abs2, x-y)
rmse(U*ũ_heat, u_heat), rmse(U2*ũ_heat2, u_heat)

By adding the second mode, we obtain a roughly 4$\times$ reduction in root-mean-square error over all grid cells and timesteps. However, the runtime is theoretically $O(r^2)$, so it would also increase 4$\times$ if we double the modes. In practice, the runtime didn't seem to increase much; I'm not sure what compiler optimizations are being made, or the runtime could be affected by spurious variable copies or something that is masking the true speed. (By the way, in general we don't expect a doubling of the modes to reduce the error by 4; the error reduction depends on the singular value spectrum, the dynamics of the system, etc.)

### A note on FOM vs. ROM computational complexity for PDEs

Galerkin projection reduces a $N\times N$ linear ODE system, which requires $O(N^2)$ computational work per timestep to do the matrix-vector multiplication, to a $r \times r$ system requiring $O(r^2)$ work. It theoretically should be efficient if $r^2\ll N^2$, or $r\ll N$.

Note, however, that PDE systems are *sparse* when discretized into ODEs, as the diffusion equation was: there are only $O(N)$ nonzero entries (the diffusion equation $A$ matrix was tridiagonal), and the sparse matrix-vector multiplication takes $O(N)$ work. This is because PDE systems are *local*: after spatial discretization, grid cells only couple to their neighbors. At each timestep, updating each grid cell's state requires a small, constant amount of computation, calculating differential operators over the "stencil" neighborhood of the cell.

If the FOM matrix $A$ is sparse, the ROM matrix $\tilde{A}$ is still usually dense: all POD modes couple to each other. This means we are going from a $O(N)$ FOM to a $O(r^2)$ ROM. The ROM of a PDE will be effective when $r^2\ll N$, or $r\ll\sqrt{N}$. This is not quite as big a win as we might have hoped for, but the ROM is still advantageous for very large $N$, for sufficiently high-resolution PDE-based models.

## Nonlinear model reduction

Next, we would like to reduce the nonlinear Allen-Cahn equation. But how? Let us revisit POD-Galerkin projection for a nonlinear system.

Suppose the ODE system is completely nonlinear,
$$\frac{\partial u}{\partial t} = f(u)$$
for some nonlinear function $f(u)$.

We can repeat the Galerkin projection procedure, by expanding the state vector in the POD basis, $u = U\tilde{u}$, and then projecting both sides of the equation into the reduced space with $U'$:
$$U'\frac{\partial (U\tilde{u})}{\partial t} = U'f(U\tilde{u})$$
We can still pull $U$ out of the derivative and cancel it with $U'$ by orthonormality, leaving
$$\frac{\partial\tilde{u}}{\partial t} = U'f(U\tilde{u})$$
The problem is that, unlike the linear operator $f = A$ which gave a reduced operator $U'AU$, we can't precompute a reduced-dimension operator $\tilde{f}$: the operation $U'f(U\tilde{u})$ involves lifting $\tilde{u}$ to the full space with $U\tilde{u}=u$, evaluating $f(\cdot)$ in the $N$-dimensional full state space, then projecting back down to the reduced space with $U'$.

If the system is semilinear as in Allen-Cahn,
$$\frac{\partial u}{\partial t} = Au + f(u)$$
then we can perform the linear Galerkin projection on $A$, but we're still stuck with the nonlinear $f$ piece being just as expensive as the FOM.

Nevertheless, this works *formally*, so let's demonstrate that it works. Since we changed the FOM from diffusion to Allen-Cahn, we'll have to construct a new POD basis from the Allen-Cahn solution snapshots, and thus a new $\tilde{A}$ matrix; even though the linear operator $A$ (the Laplacian) is the same in both the diffusion and Allen-Cahn equations, $\tilde{A}$ is different because we're projecting onto a different solution basis.

In [ ]:
function rom_allen_cahn_matrix!(dũ, ũ, t, (Ã,f,U)=p)
    dũ .= Ã*ũ + U'*f.(U*ũ)
end

In [ ]:
U_SVD_ac,_,_ = svd(u_ac)
r = 1
U = U_SVD_ac[:,1:r]
Ã = U'*(A*U)

In [ ]:
@time ũ_ac = solve_euler(rom_allen_cahn_matrix!, U'u₀, ts, (Ã,f,U))
@time ũ_ac = solve_euler(rom_allen_cahn_matrix!, U'u₀, ts, (Ã,f,U))

@time @gif for i = 1:length(ts)
    plot(x, u_ac[:,i], title="Allen-Cahn equation", ylim=(0.0, maximum(u₀)+1e-3), lw=3, label="FOM")
    plot!(x, U*ũ_ac[:,i], ls=:dash, lw=3, label="ROM")    
end every 1024

As we can see from the timing, the ROM is a bit faster (~5$\times$?) than the FOM was, but not drastically so. The speedup comes from solving the linear part faster, even if the nonlinear part is as expensive as it was before.

Let's try adding another POD mode.

In [ ]:
r = 2
U2 = U_SVD_ac[:,1:r]
Ã2 = U2'*(A*U2)
@time ũ_ac2 = solve_euler(rom_diffusion_matrix!, U2'u₀, ts, Ã2)
@time ũ_ac2 = solve_euler(rom_diffusion_matrix!, U2'u₀, ts, Ã2)

@time @gif for i = 1:length(ts)
    plot(x, u_ac[:,i], title="Allen-Cahn equation", ylim=(0.0, maximum(u₀)+1e-3), lw=3, label="FOM")
    plot!(x, U*ũ_ac[:,i], ls=:dash, lw=3, label="ROM (1 mode)")
    plot!(x, U2*ũ_ac2[:,i], ls=:dash, lw=3, label="ROM (2 modes)")
end every 1024

Unlike the diffusion equation, adding a second mode to the Allen-Cahn equation produces a large ROM error! How can this happen, when adding modes decreases the POD approximation error?

Well, let's first verify that adding modes *does* decrease the POD error: we will project and then reconstruct the FOM solution.

In [ ]:
@time @gif for i = 1:length(ts)
    plot(ylim=(-1e-2,1e-2), legend=:topleft, title="Allen-Cahn equation POD reconstruction error", ylabel="Reconstruction residual", lw=3)
    plot!(x, U*(U'*u_ac[:,i]) - u_ac[:,i], ls=:dash, label="1 POD mode", lw=3)
    plot!(x, U2*(U2'*u_ac[:,i]) - u_ac[:,i], ls=:dash, label="2 POD modes", lw=3)
end every 1024

2 POD modes do seem to be a better approximation to the FOM solution than 1 mode. So why is the ROM worse? It is known that very high POD modes that contain little variance or "power" can be "polluted" by numerical errors or "noise" in the SVD. In our case, all the modes beyond the first are very, very small due to the exponential decay of the Kolmogorov $n$-width for diffusion-like equations. I hypothesize that the nonlinearity of Allen-Cahn is amplifying the noise in higher-order modes in a way that does not affect the linear diffusion equation.

## Hyperreduction

Let us return to the central problem of nonlinear projection-based model reduction, which is that the nonlinearity prevents us from working entirely in the reduced state space: we always have to evaluate the nonlinear RHS on the full-order state vector and then project it back down to a reduced state vector, which defeats the purpose of using a ROM in the first place.

What we really need is a way to evaluate the nonlinearity entirely in the reduced space, without ever lifting back to the full-order space. This is esssential to making ROMs useful in most real-world settings. There are various ways to do it, which collectively go by the name of "hyperreduction". We will study one particular hyperreduction method.

(Incidentally, for the rest of this notebook we will set aside the last section's counterintuitive problem of ROM errors increasing as POD solution approximation quality decreases, since this may be more of an exception due to the extremely high quality of the leading POD mode for our test case.)

### Discrete empirical interpolation method (DEIM)

The key idea in DEIM is that we don't want to evaluate the nonlinear RHS at all $N$ grid cells in the FOM state space, because $N$ is high-dimensional. Instead, we are going to select a subset of $m$ grid cells at which to evaluate the nonlinear function, called *DEIM points*, and use this data to *approximate* the action of the nonlinearity on the full $N$-dimensional state vector. If $m\ll N$, then we are back in business for building useful ROMs. We will explain shortly how the DEIM points may be selected, but for now we will assume they have been just handed to us from on high (in fact, practitioners sometimes refer to them as "magic points"). In this exposition, suppose we have been handed $m=3$ DEIM points, corresponding to the grid cells 3, 47, and 226 (in code, we store them as a list of indices into the state vector, `p = [3, 47, 226]`).

To see how to use this for hyperreduction, let us return to the general nonlinear FOM,
$$\frac{\partial u}{\partial t} = f(u)$$

Since basis expansion worked so well for us in building ROMs, we're going to use it again. Let's assume that the $N$-dimensional vector $f(u)$ &mdash; which is expensive to compute &mdash; can be expanded in a basis $\{V_1,\ldots,V_m\}$, which can be thought of a $N\times m$ matrix $V$ whose columns are the $\{V_i\}$. Then
$$f(u) \approx \sum_{i=1}^m c_i V_i$$
or, in matrix form,
$$f(u) \approx Vc$$
for some $m$-dimensional coefficient vector $c\equiv c(u)$ which, importantly, depends on $u$. (Right now we're also not going to worry about where the basis $V$ comes from.)

To determine the coefficients $c$, we require that the above approximation become *exact* at the locations of the DEIM points, which are select grid cells in the FOM domain. Let the $m$-dimensional vector $u_p$ be the restriction of $u$ to the locations of $m$ grid cells (DEIM points), and $f(u_p)$ be the nonlinearity $f$ evaluated at those grid cells. To compute the basis expansion at the DEIM points, we only need the basis vector elements at the DEIM point locations as well, so the $m\times m$ matrix $V_p$ consists of the rows of $V$ corresponding to the $m$ grid cells.

In code, we compute $u_p$ with `up = u[p]` and $V_p$ with `Vp = V[p,:]`.

DEIM requires require that the basis expansion hold at the DEIM points exactly:
$$f(u_p) = V_p c$$
This is related to the concept of *collocation points* in numerical PDEs, in which the solver method is constructed so that the solution satisfies the PDE at a chosen set of points in the domain.

We can then solve for the coefficients $c = V_p^{-1} f(u_p)$, or in code, `c = Vp \ f.(up)`, where the `\` backsolve operator is used to solve linear systems (i.e. `A\b` solves the linear system `Ax = b`).

The final missing piece is that we have been speaking of selecting $m$ elements out of the $N$-dimensional state vector $u$. But in a ROM, we don't want to reconstruct this high-dimensional vector; we want to work direction with the $r$-dimensional reduced state vector $\tilde{u}$. The two are related by $u = U\tilde{u}$. However, like with $V$, we don't have to work with the full matrix $U$. We can construct a matrix $U_p$ consisting of the $m$ rows of $U$ corresponding to the DEIM points, or `Up = U[p,:]`. Then $u_p = U_p\tilde{u}$.

Putting this all together: we previously found that we could formally construct a nonlinear ROM of the form
$$\frac{\partial\tilde{u}}{\partial t} = U'f(U\tilde{u})$$
but it was impractical because it requires evaluating the high-dimensional $f(u)\equiv f(U\tilde{u})$. Now, using DEIM hyperreduction, we can rewrite the nonlinearity in the RHS in terms of DEIM points:
$$U'f(U\tilde{u}) \approx U'(Vc) = U'(V(V_p^{-1} f(u_p))) = U'(V(V_p^{-1} f(U_p\tilde{u})))$$
If we precompute $W = U'V$, we finally arrive at
$$\frac{\partial\tilde{u}}{\partial t} = W V_p^{-1} f(U_p\tilde{u})$$
or if we also precompute $B=W V_p^{-1}$ (but see later discussion for nuances of explicitly computing matrix inverses),
$$\frac{\partial\tilde{u}}{\partial t} = B f(U_p\tilde{u})$$
If we are reducing a semilinear equation,
$$\frac{\partial u}{\partial t} = Au + f(u)$$
we can apply Galerkin projection to the linear term as before, and DEIM to the nonlinear term:
$$\frac{\partial\tilde{u}}{\partial t} = \tilde{A}\tilde{u} + W V_p^{-1} f(U_p\tilde{u}) = \tilde{A}\tilde{u} + B f(U_p\tilde{u})$$
where $\tilde{A} = U^{\mathsf{T}} A U$ as before.

The point was to approximate the nonlinearity using linear algebra with precomputed, low-dimensional matrices: $W$ (or $B$) is $r\times m$, $V_p$ is $m\times m$, and $U_p$ is $m\times r$; the ROM is successful when both $r$ and $m$ are small.

This derivation entailed some tricky matrix manipulation, but the final DEIM code is deceptively simple:

In [ ]:
function deim_matrices(U, V, p)
    (Up, Vp, W) = U[p,:], V[p,:], U'V
end

function rom_deim!(dũ, ũ, t, (Ã,f,Up,Vp,W)=p)
    dũ .= Ã*ũ + W * (Vp \ f.(Up*ũ))
end

*Advanced implementation note:* Whenever we timestep the ROM we have to compute a matrix inverse $V_p^{-1}$, so we might want to actually precompute $V_p^{-1}$ instead of $V_p$ so we don't have to re-invert the same matrix every timestep, which would be an $O(m^3)$ operation. However, numerically, it's not always good to compute inverses of matrices directly (as with `inv(V[p,:])`); that can be numerically unstable and also sometimes slower than it needs to be.

If you want to compute the product of a matrix inverse and a vector, $M^{-1} v$, the recommended practice is usually to *factorize* the matrix, e.g. into lower and upper triangular factors $M = LU$ ("LU-decomposition"). The factorization, which you pre-compute like you would the inverse, still requires $O(m^3)$ floating-point operations. Once you have it, you can efficiently solve the two equations $y = L^{-1}v$ and $x = U^{-1}y$ to get $M^{-1}v = U^{-1}(L^{-1}v)$ in $O(m^2)$ time, as fast as multiplying a (pre-computed inverse) matrix with a vector. However, it's more numerically stable than working with the matrix inverse. The Julia `\` operator will operate on a factorized matrix object to perform these forward and backsolve steps, `lu(M)\v`, the same way it will solve a linear system $M^{-1}v$ with `M\v`.

In fact, `factorize` in Julia will attempt to intelligently select which factorization (LU, QR, or other) is best for your matrix and return an appropriate factorization object.

See also [J.D. Cook, "Don't Invert that Matrix"](https://www.johndcook.com/blog/2010/01/19/dont-invert-that-matrix/) and [Druinsky and Toledo, "How Accurate is `inv(A)*b`?"](https://arxiv.org/abs/1201.6035) for a counterpoint.

In [ ]:
function deim_matrices(U, V, p)
    (Up, Vpf, W) = U[p,:], factorize(V[p,:]), U'V
end

function rom_deim!(dũ, ũ, t, (Ã,f,Up,Vp,W)=p)
    dũ .= Ã*ũ + W * (Vpf \ f.(Up*ũ))
end

However, as mentioned earlier, however, if the nonlinear term in the ROM RHS is $W V_p^{-1} f(U_p\tilde{u})$, we can precompute the entire matrix $B = W V_p^{-1}$ and just compute $B f(U_p\tilde{u})$ in the RHS.  The numerically stable way of writing $B = W V_p^{-1}$ with backsolves is `B = (V[p,:]'\(U'V)')' = (V[p,:]'\(V'U))'`, since
$$
\begin{align}
B' &= (U'VV_p^{-1})'\\
   &= (V_p^{-1})' (U'V)'\\
   &= ({V_p}')^{-1} (U'V)'\\
   &= ({V_p}') \backslash (U'V)'
\end{align}
$$
so
$$
\begin{align}
B &= (({V_p}') \backslash (U'V)')'\\
 &= (({V_p}') \backslash (V'U))'\\
\end{align}
$$

In [ ]:
function deim_matrices(U, V, p)
    (Up, B) = U[p,:], (V[p,:]'\(V'U))'
end

function rom_deim!(dũ, ũ, t, (Ã,f,Up,B)=p)
    dũ .= Ã*ũ + B*f.(Up*ũ)
end

#### A note on nonlinear FOM vs. ROM computational complexity with DEIM hyperreduction

Earlier we saw that reduction of a linear FOM
$$\frac{\partial u}{\partial t} = Au$$
of state vector dimension $N$ to to a linear ROM
$$\frac{\partial \tilde{u}}{\partial t} = \tilde{A}\tilde{u}$$
of state vector dimension $r$ corresponds to going from $O(N)$ work to $O(r^2)$, assuming the FOM operator $A$ is sparse and only contains $O(N)$ nonzero elements rather than $O(N^2)$.

In the semilinear setting, the FOM is
$$\frac{\partial u}{\partial t} = Au + f(u)$$
where each term requires $O(N)$ work (since $A$ is sparse and $f(\cdot)$ is assumed to act pointwise/locally on each grid cell, so the FOM is still $O(N)$.

The ROM with DEIM hyperreduction is
$$\frac{\partial \tilde{u}}{\partial t} = \tilde{A}\tilde{u} + B f(U_p \tilde{u})$$
The hyperreduced term requires $O(mr)$ work to lift to the DEIM points with $U_p\tilde{u}$, $O(m)$ work to evaluate $f$ at those points, and $O(mr)$ work to left-multiply the nonlinearity by $B$.

The FOM is thus $O(N)$ while the ROM is $O(r^2) + O(mr)$, coming from the linear and nonlinear terms respectively. The linear term would appear to dominate, *unless* evaluating the nonlinearity is very expensive, in which case $O(mr)$ will dominate.

In the linear case or semilinear case when the nonlinearity is cheap, the ROM shows speedup of order $N/r^2$ when $r\ll\sqrt{N}$. When the nonlinear term is expensive, the ROM shows speedup of order $N/(mr)$ when $mr\ll N$. This is roughly the same as in the linear case if $m$ and $r$ are of the same order.

#### DEIM basis

We are in the home stretch here. The last pieces of the puzzle are (1) where did the basis $V$ come from in which we expanded $f(u)$, and (2) where do the "magic" DEIM points come from?

The answer to the first question, of course, is more POD. The $U$ basis that we used to reduce the state vectors was computed from snapshot data consisting of solution states. The DEIM basis $V$ that we use to reduce the nonlinear term will be computed via POD from snapshots of $f(u)$. If our equation is fully nonlinear, $\partial u/\partial t = f(u)$, that means we are applying POD to the FOM *tendencies* or time derivatives $\partial u/\partial t$; if it is semilinear, $\partial u/\partial t = Au + f(u)$, then we have to pull the $f(u)$ function out from the RHS and save its output.

Either way, we usually have to "intrusively" modify our solver to return the $f(u)$ data to needed to construct the DEIM basis, as well as the solution $u$. This is one reason why ROMs are not always used in practice for complex production codes. For our Allen-Cahn solver, it's not so hard.

In [ ]:
function fom_allen_cahn_deim!(du, fu, u, t, (A,f)=p)
    fu .= f.(u)
    du .= A*u + fu
end

In [ ]:
function solve_euler_deim(rhs_deim!, u₀, ts, p)
    N = length(u₀)
    Nsteps = length(ts)
    Δt = ts[2]-ts[1]
    u = zeros(N, Nsteps)
    u[:,1] = u₀
    duₜ = zeros(N)
    fu = zeros(N, Nsteps-1)
    fuₜ = zeros(N)
    
    for t = 1:Nsteps-1
        uₜ = @view u[:,t]
        rhs_deim!(duₜ, fuₜ, uₜ, ts[t], p)
        u[:,t+1] = u[:,t] + duₜ * Δt
        fu[:,t] = fuₜ
    end

    u, fu
end

In [ ]:
@time u_ac_deim, f_ac_deim = solve_euler_deim(fom_allen_cahn_deim!, u₀, ts, (A,f));

In [ ]:
DU,DS,DV = svd(f_ac_deim) # DEIM basis U,S,V matrices
V = DU # basis V is left singular vectors as usual in POD

plot(S, yscale=:log10, xlim=(1,10), ylim=(1e-3,1e4), xticks=1:10, lw=3, xlabel="POD basis vector", ylabel="Singular value", title="Singular values of Allen-Cahn f(u) snapshots", label="")

Like the $U$ POD modes, the $V$ POD modes decay rapidly after the first one, suggesting that we won't need many DEIM points.

#### Greedy DEIM point selection

We return, at last, to the question of how to compute the DEIM points. This is usually done by a "greedy" algorithm that selects the next point one-at-a-time according to an optimality principle, without regard to whether all the points as a whole are collectively optimal.

The first DEIM point is easy. We select the element (grid cell) of the first DEIM basis vector $V_1$ (i.e. POD mode 1) that has the largest magnitude
$$p_1 = \mathop{\operatorname{arg\,max}}_i\,|(V_1)_i|$$
on the grounds that the first POD mode explains most of $f(u)$, and the point in the domain where that mode is largest is the most informative about what $f(u)$ might be over the whole domain.

The rest of the DEIM points are defined recursively and computed iteratively. From each POD mode we select a single DEIM point. However, we don't just choose the points where each mode is largest, like we did for the first point.

Instead, assume we have selected the first $k$ DEIM points, $\{p_1,\ldots,p_k\}$, from the first $k$ POD modes $\{V_1,\ldots,V_k\}$, which we represent by a $N\times k$ basis matrix $V_{(1:k)}$ (`Vk = V[:,1:k]`). The $k+1$th DEIM point is the one that has the greatest *residual error* when trying to expand mode $V_{k+1}$ in terms of modes $\{V_1,\ldots,V_k\}$ using DEIM.

That is:
1. Expand the $k+1$th mode in terms of the first $k$ modes: $V_{k+1} \approx V^{(k)} c$
2. Solve for the coefficients $c = ({V_{(1:k)}}_p)^{-1} (V_{k+1})_p$, where as before we restrict the matrices (rows) to the locations of the (current) DEIM points (`Vkp = Vk[p,:]` and `V[p,k+1]`, where `p` is actually `p[1:k]`).
3. Compute the residual error between the basis expansion prediction for $V_{k+1}$ and the actual $V_{k+1}$, $r =  V_{(1:k)} c - V_{k+1} = V_{(1:k)} ({V_{(1:k)}}_p)^{-1} (V_{k+1})_p - V_{k+1}$.
4. The next DEIM point is the element of the residual (grid cell) with the largest error: $p_{k+1} = \mathop{\operatorname{arg\,max}}_i\,|(V_1)_i|$.

As before, this seems complicated, and there is some juggling indices into the matrices and such, but it leads to a simple implementation.

In [ ]:
function deim_points(V, m)
    p = zeros(Int, m)
    p[1] = argmax(abs.(V[:,1]))
    for k = 1:m-1
        c = V[p[1:k], 1:k] \ V[p[1:k], k+1]
        r = V[:,1:k]*c - V[:,k+1]
        p[k+1] = argmax(abs.(r))
    end
    p
end

In [ ]:
m = 5
p = deim_points(V, m)

Let's visualize the DEIM point selection process, point by point. The first point is just where mode 1 has the largest absolute magnitude.

In [ ]:
plot(x, V[:,1], legend=:top, label="Mode 1", lw=3, title="V mode 1")
vline!([x[p[1]]], lw=2, ls=:dash, label="DEIM point 1")

The remaining DEIM points are where the difference between the predicted and actual mode is greatest in absolute magnitude.

In [ ]:
k = 1
c = V[p[1:k], 1:k] \ V[p[1:k], k+1]

plot(x, V[:,k+1], legend=:bottomright, lw=3, title="V mode $(k+1)", label="POD mode $(k+1)")
vline!([x[p[k+1]]], lw=2, ls=:dash, label="DEIM point $(k+1)")
plot!(x, V[:,1:k]*c, lw=3, label="Mode $(k+1) predicted from modes <$(k+1)")

In [ ]:
k = 2
c = V[p[1:k], 1:k] \ V[p[1:k], k+1]

plot(x, V[:,k+1], legend=:top, lw=3, title="V mode $(k+1)", label="POD mode $(k+1)")
vline!([x[p[k+1]]], lw=2, ls=:dash, label="DEIM point $(k+1)")
plot!(x, V[:,1:k]*c, lw=3, label="Mode $(k+1) predicted from modes <$(k+1)")

In [ ]:
k = 3
c = V[p[1:k], 1:k] \ V[p[1:k], k+1]

plot(x, V[:,k+1], legend=:bottom, lw=3, title="V mode $(k+1)", label="POD mode $(k+1)")
vline!([x[p[k+1]]], lw=2, ls=:dash, label="DEIM point $(k+1)")
plot!(x, V[:,1:k]*c, lw=3, label="Mode $(k+1) predicted from modes <$(k+1)")

In [ ]:
k = 4
c = V[p[1:k], 1:k] \ V[p[1:k], k+1]

plot(x, V[:,k+1], legend=:bottom, lw=3, title="V mode $(k+1)", label="POD mode $(k+1)")
vline!([x[p[k+1]]], lw=2, ls=:dash, label="DEIM point $(k+1)")
plot!(x, V[:,1:k]*c, lw=3, label="Mode $(k+1) predicted from modes <$(k+1)")

## Putting it all together: POD-Galerkin-DEIM ROM for a nonlinear FOM

We have spread the code throughout the notebook, so let's collect it all in one place to build a FOM and ROM for the nonlinear Allen-Cahn equation.

### Full Order Model (FOM)

In [ ]:
T = 1.0
Nsteps = 65536
ts = LinRange(0, T, Nsteps)
Δt = ts[2]-ts[1]

L = 1.
Nx = 512
Δx = L/Nx

x = LinRange(0, L, Nx)
u₀ = sin.(π*x)

κ = 0.1

In [ ]:
A = (κ / Δx^2) * spdiagm(-1 => fill(1.0,Nx-1), 0 => fill(-2.0,Nx), 1 => fill(1.0,Nx-1))

f(u) = u^3 - u

function fom_allen_cahn_deim!(du, fu, u, t, (A,f)=p)
    fu .= f.(u)
    du .= A*u + fu
end

*Timestepper*:

1. Normal version that outputs solution trajectory
2. DEIM-intrusive version that also outputs nonlinearity data given a RHS that computes this separately

In [ ]:
function solve_euler(rhs!, u₀, ts, p)
    N = length(u₀)
    Nsteps = length(ts)
    Δt = ts[2]-ts[1]
    u = zeros(N, Nsteps)
    u[:,1] = u₀
    duₜ = zeros(N)
    
    for t = 1:Nsteps-1
        uₜ = @view u[:,t]
        rhs!(duₜ, uₜ, ts[t], p)
        u[:,t+1] = u[:,t] + duₜ * Δt
    end

    u
end

In [ ]:
function solve_euler_deim(rhs_deim!, u₀, ts, p)
    N = length(u₀)
    Nsteps = length(ts)
    Δt = ts[2]-ts[1]
    u = zeros(N, Nsteps)
    u[:,1] = u₀
    duₜ = zeros(N)
    fu = zeros(N, Nsteps-1)
    fuₜ = zeros(N)
    
    for t = 1:Nsteps-1
        uₜ = @view u[:,t]
        rhs_deim!(duₜ, fuₜ, uₜ, ts[t], p)
        u[:,t+1] = u[:,t] + duₜ * Δt
        fu[:,t] = fuₜ
    end

    u, fu
end

*Generate training data for POD*:

(for $U$ and $V$ basis matrices)

In [ ]:
@time u_ac_fom, f_ac_fom = solve_euler_deim(fom_allen_cahn_deim!, u₀, ts, (A,f));
@time u_ac_fom, f_ac_fom = solve_euler_deim(fom_allen_cahn_deim!, u₀, ts, (A,f));

### Reduced Order Model (ROM)

*DEIM point selection:*

In [ ]:
function deim_points(V, m)
    p = zeros(Int, m)
    p[1] = argmax(abs.(V[:,1]))
    for k = 1:m-1
        c = V[p[1:k], 1:k] \ V[p[1:k], k+1]
        r = V[:,1:k]*c - V[:,k+1]
        p[k+1] = argmax(abs.(r))
    end
    p
end

*Precompute all the matrices the ROM will need:*

In [ ]:
function build_rom(A, f, u, fu, r, m)
    U = svd(u).U[:,1:r] # POD basis for state snapshots, retain r basis vectors

    Ã = U'*(A*U) # ROM linear operator
    
    V = svd(fu).U[:,1:m] # POD basis for DEIM f(u) snapshots, retain m basis vectors
    p = deim_points(V, m) # compute DEIM points and matrices
    Up = U[p,:]
    Vp = V[p,:]
    B = (Vp'\(V'U))'
    
    params = (U=U, V=V, Ã=U'*(A*U), f=f, p=p, Up=Up, Vp=Vp, B=B)
end

*POD-Galerkin-DEIM ROM RHS:*

In [ ]:
function rom_deim!(dũ, ũ, t, (U,V,Ã,f,p,Up,Vp,B)=p)
    dũ .= Ã*ũ + B*f.(Up*ũ)
end

*Build the ROM matrices:*

### Running and testing the ROM

In [ ]:
r = 1
m = 1

@time (U,V,Ã,f,p,Up,Vp,B) = params = build_rom(A, f, u_ac_fom, f_ac_fom, r, m);

*Solve the ROM:*

In [ ]:
@time ũ_ac_rom = solve_euler(rom_deim!, U'*u₀, ts, params)
@time ũ_ac_rom = solve_euler(rom_deim!, U'*u₀, ts, params)
u_ac_rom = U * ũ_ac_rom;

In [ ]:
@time @gif for i = 1:length(ts)
    plot(x, u_ac_fom[:,i], title="Allen-Cahn equation", ylim=(0.0, maximum(u₀)+1e-3), lw=3, label="FOM")
    plot!(x, u_ac_rom[:,i], ls=:dash, lw=3, label="ROM")    
end every 1024

Using only 1 snapshot POD mode and 1 DEIM point, on my computer I got about $20\times$ speedup while maintaining reasonable accuracy in the eyeball norm. The speedup factor should increase for higher-resolution simulations (more grid cells), because the FOM becomes more expensive but the ROM typically won't require many (if any) more POD modes or DEIM points for a system of this type. (This is not universally true.)

You can experiment with changing the number of modes and DEIM points, and properly we should be doing some formal residual error checks against the FOM to evaluate ROM quality.

#### DEIM check

We can also check to see that the how well DEIM is approximating $f(u=U\tilde{u})$ and whether it is being calculated exactly at the DEIM points, as designed.

In [ ]:
@time @gif for i = 1:length(ts)
    fu = f.(U * ũ_ac_rom[:,i])
    fu_DEIM = V[:,1:m] * (Vp \ f.(Up*ũ_ac_rom[:,i]))
    
    plot(x, fu, title="Allen-Cahn nonlinear term", ylim=(-0.45,0), lw=3, label="f(u): true", legend=:right)
    plot!(x, fu_DEIM, lw=3, ls=:dash, label="f(u): DEIM")
    scatter!(x[p], fu_DEIM[p], ms=6, label="f(u) @ DEIM points")       
end every 1024

We see the DEIM approximation is not perfect, especially at the early states, but this didn't seem to cause a big problem for the ROM solution. That won't always be the case for more challenging systems.

Even one more DEIM point ($m=2$) helps a lot for approximating the Allen-Cahn equation's cubic nonlinearity:

In [ ]:
r = 1
m = 2

@time (U,V,Ã,f,p,Up,Vp,B) = params = build_rom(A, f, u_ac_fom, f_ac_fom, r, m)
@time ũ_ac_rom = solve_euler(rom_deim!, U'*u₀, ts, params)
@time ũ_ac_rom = solve_euler(rom_deim!, U'*u₀, ts, params)
u_ac_rom = U * ũ_ac_rom

@time @gif for i = 1:length(ts)
    fu = f.(U * ũ_ac_rom[:,i])
    fu_DEIM = V[:,1:m] * (Vp \ f.(Up*ũ_ac_rom[:,i]))
    
    plot(x, fu, title="Allen-Cahn nonlinear term", ylim=(-0.45,0), lw=3, label="f(u): true", legend=:right)
    plot!(x, fu_DEIM, lw=3, ls=:dash, label="f(u): DEIM")
    scatter!(x[p], fu_DEIM[p], ms=6, label="f(u) @ DEIM points")       
end every 1024

Re-solving with $m=2$ DEIM points for completeness:

In [ ]:
@time ũ_ac_rom = solve_euler(rom_deim!, U'*u₀, ts, params)
@time ũ_ac_rom = solve_euler(rom_deim!, U'*u₀, ts, params)
u_ac_rom = U * ũ_ac_rom

@time @gif for i = 1:length(ts)
    plot(x, u_ac_fom[:,i], title="Allen-Cahn equation", ylim=(0.0, maximum(u₀)+1e-3), lw=3, label="FOM")
    plot!(x, u_ac_rom[:,i], ls=:dash, lw=3, label="ROM")    
end every 1024

## Discussion and Outlook

We conclude with a discussion of what we've done, take a step back to the bigger picture of why and when this is useful and what the caveats are, and how to go beyond the basic POD-Galerkin-DEIM PROM.

### What was the point?

We saw that we could speed up the solution of a model by projecting its governing equations onto a reduced basis. This seems highly useful!

... *however*. In order to build the reduced basis, we needed to compute a data-driven POD of a solution trajectory. So from another perspective, this looks pointless: solving an expensive FOM to produce a ROM ... that just reproduces the FOM solution we already had.

The point, of course, is that we'd like to use the ROM to quickly generate *different* solutions than the one(s) we used to construct the ROM (i.e. its POD basis). For example, we might want to change the initial conditions, or boundary conditions, or parameters of the FOM and get a fast solution to the new system.

All of this is possible in principle: we can give the solver a different initial state than the one used to generate the POD snapshots, or we could add a forcing term and project that onto the reduced basis, or change a parameter (like $\kappa$ in the diffusion or Allen-Cahn equations); this would require recomputing some ROM matrices every time we change something, but the ROM might be fast enough that the precompute + ROM solve is still faster than a new FOM solve.

### Will it work?

A major sticking point is that the ROM is only as good as its POD basis. In our examples, we used a sine-wave initial state, and the leading POD mode captures much of the shape of this state. If we gave the ROM a very different initial state, the POD basis derived from a sine-wave initial state may no longer capture the new solution's trajectory using a low number of modes.

In addition, sometimes POD simply fails to reduce the state space at all. This is common in wave equations like the advection equation. Consider a dynamics where, instead of diffusing, a sine wave state propagates at some constant velocity across the domain. The state at any given time can't be captured by any *fixed* basis, because it's moving: we'd need basis vectors at every location where the solution could be in a given snapshot. The Kolmogorov $n$-width does not decay at all &mdash; all modes are equally important.

Nevertheless, ROMs can be useful. The POD basis is usually not trained only on snapshots from a single FOM solve, but from a variety of solves under different conditions (parameter settings, initial conditions, etc.). More modes will be required to describe all these different types of solutions, but for some systems the ROM still comes out ahead.

One might ask why we wouldn't just use statistical machine learning emulators, like neural networks or Gaussian processes, at this point, since we need an ensemble of training data either way (to fit the statistical model, or to construct a ROM basis). One answer is that the training data is *only* needed by the ROM for dimension reduction, which may require less data than response-surface statistical regression methods that need to predict how the entire solution varies across the ensemble. The ROM takes care of all the predictions once the low-dimensional data subspace is constructed by POD.

### Outlook

Modern ROM research has focused on a variety of areas to make ROMs more useful, e.g. to extrapolate better with less data. There is work on basis expansions or other (potentially nonlinear or ML-based) dimension reduction methods besides POD, and the basis can be adaptive. There are other hyperreduction methods besides DEIM. ROM "closure" schemes attempt to correct for POD truncation error: because we only have a limited number of modes with which to describe the solution, some of the solution can get lost into "unresolved modes", and "closure schemes" try to keep track of and account for the average behavior of the features of the solution that are unresolved (orthogonal to all the POD modes being used). ROMs can become unstable, for this and other reasons (they may not conserve energy if you're resolving all modes, for example), so there are related ROM stabilization methods. More generally, "structure-preserving" ROMs are constructed so that the ROM exactly preserves properties of the FOM, such as conservation laws or symmetries. There are other active learning, experimental design, and multi-fidelity types of FOM-ROM workflows in which error metrics are used to identify when the ROM is breaking down and additional FOM solves are needed as snapshot training data or to improve ROM closures. There are also many areas of cross-fertilization and hybridization of ROMs with ML techniques like neural networks (learned latent spaces for data reduction, learned dynamical operators or matrices, learned closure schemes, etc.). And, of course, there are other model reduction techniques besides projection-based model reduction (many ML surrogate methods, latent neural ODEs, operator inference, etc.).

Another pain point, and a significant reason why ROMs are not always used in engineering practice, is their "intrusive" nature: in order to project the FOM governing equations onto a reduced basis, you need to know what those equations are (for example, $A$ and $f$). Furthermore, for hyperreduction methods like DEIM, you need access to data like nonlinearity snapshots ($f(u)$), that aren't normally saved out by a FOM code, requiring code modification to the simulator. When models are extremely complicated and their equations are not analytically tractable, or the codebase is not easily modified, such ROMs may be impractical. One approach to circumvent this problem is to apply *system identification*, an inverse method that attempts to recover the governing equations of a dynamical system from its solution data. This allows us to convert intrusive PROM methods into non-intrusive or "black box" methods that do not require access to the underlying system equations or model source code. For one example of this (with applications to model structural uncertainty / model-form error, i.e. uncertainty in the governing equations), see [DeGennaro, Urban, Nadiga, and Haut, "Model structural inference using local dynamic operators",  _International Journal of Uncertainty Quantification_ **9**, 59 (2019)](https://dx.doi.org/10.1615/Int.J.UncertaintyQuantification.2019025828).

### Coda

In the spirit of code golf ("lowest score [=lines of code] wins" in golf), below is a compact version of the entire POD-Galerkin-DEIM reduced order model (ROM) algorithm, if we take all the other standard ODE solver technology as given, to reduce the general semilinear ODE system (FOM)
$$\frac{\partial u}{\partial t} = Au + f(u)$$
where $A$ is a linear operator (matrix) and $f(\cdot)$ is a pointwise nonlinear term. In the code, $r$ and $m$ are the reduced state dimension and number of DEIM points, and `u` and `fu` are snapshot training data of the solution $u(x,t)$ and ODE nonlinearity $f(u(x,t))$ from one or more solutions of the FOM.

In [ ]:
function build_rom(A, f, u, fu, r, m)
    U, V = svd(u).U[:,1:r], svd(fu).U[:,1:m]
    p = vcat(argmax(abs.(V[:,1])), zeros(Int,m-1))
    for k = 1:m-1
        p[k+1] = argmax(abs.(V[:,1:k]*(V[p[1:k],1:k]\V[p[1:k],k+1]) - V[:,k+1]))
    end
    (U=U, V=V, Ã=U'*(A*U), f=f, p=p, Up=U[p,:], Vp=V[p,:], B=(V[p,:]'\(V'U))')
end

rom_rhs!(dũ, ũ, (;U,V,Ã,f,p,Up,Vp,B), t) = dũ .= Ã*ũ + B*f.(Up*ũ)

rom_rhs! (generic function with 1 method)

Just remember to left-multiply by $U'$ to project the initial state for the solver ($\tilde{u}_0 = U'u_0$), and left-multiply by $U$ to reconstruct the solver's solution ($u = U\tilde{u}$), and everything else can leverage standard differential equation libraries.

Paraphrasing an old parable, it's trivial to write 10 lines of code, but knowing *which* 10 lines of code to write is worth a lot.

# Miles' Code

In [4]:
# just a fom rhs
fom_rhs!(du,u,(;A,f),t) = du .= A*u+f.(u)

fom_rhs! (generic function with 1 method)

In [ ]:
# Dr. Urban's ROM (hyperreduction included) code

function build_rom(A, f, u, fu, r, m)
    # POD modes from trajectory data (build trial subspace) and nonlinearity evaluations (DEIM) resp.
    U, V = svd(u).U[:,1:r], svd(fu).U[:,1:m]

    # build an empty vector of DEIM points; the first one is the max norm point
    # on the first hyperreduction POD mode
    p = vcat(argmax(abs.(V[:,1])), zeros(Int,m-1))

    # greedy alg for selecting next DEIM point; you're going mode by mode and finding where the approximation 
    # over the first k modes of the k+1st mode is worst
    for k = 1:m-1
        p[k+1] = argmax(abs.(V[:,1:k]*(V[p[1:k],1:k]\V[p[1:k],k+1]) - V[:,k+1]))
    end

    # a NamedTuple that that builds the ingredients for running the ROM
    # note that we're pre-preparing B with Julia's solver, which will be stable
    (U=U, V=V, Ã=U'*(A*U), f=f, p=p, Up=U[p,:], Vp=V[p,:], B=(V[p,:]'\(V'U))')
end

# compact 1-line function definition
# update derivative vector 
rom_rhs!(dũ, ũ, (;U,V,Ã,f,p,Up,Vp,B), t) = dũ .= Ã*ũ + B*f.(Up*ũ)


rom_rhs! (generic function with 1 method)